# BiLSTM — Is This a Good Product?
**Dataset:** Fake product review scores (no download needed)
**Input:** quality, price, delivery, support scores | **Output:** Good / Bad

> BiLSTM = Bidirectional LSTM. Reads the sequence both FORWARD and BACKWARD — gets better context than regular LSTM.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout, Input

# ── Dataset ───────────────────────────────────────────────
# [quality, price_value, delivery, support] all out of 10
# 1 = Good Product, 0 = Bad Product
X = np.array([
    # Good (high scores)
    [9,8,9,9],[8,9,8,8],[9,9,9,8],[8,8,9,9],[9,8,8,9],
    [8,9,9,8],[9,8,9,8],[8,8,8,9],[9,9,8,9],[8,9,9,9],
    [9,8,9,9],[8,9,8,9],[9,9,9,9],[8,8,9,8],[9,9,8,8],
    [8,8,9,9],[9,8,8,8],[8,9,9,9],[9,9,9,8],[8,8,8,8],
    # Bad (low scores)
    [2,3,2,2],[3,2,3,3],[2,2,2,3],[3,3,2,2],[2,3,3,2],
    [3,2,2,3],[2,3,2,3],[3,3,3,2],[2,2,3,2],[3,2,2,2],
    [2,3,2,2],[3,2,3,2],[2,2,2,2],[3,3,2,3],[2,2,3,3],
    [3,3,2,2],[2,3,3,3],[3,2,2,2],[2,2,2,3],[3,3,3,3],
], dtype='float32') / 10.0   # normalize to 0-1

y = np.array([1]*20 + [0]*20)
X = X.reshape(-1, 4, 1)   # (samples, 4 timesteps, 1 feature)

# ── Train / Test Split ─────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
# ── Build BiLSTM ──────────────────────────────────────────
model = Sequential([
    Input(shape=(4, 1)),                          # 4 score features as timesteps
    Bidirectional(LSTM(16)),  # reads forward AND backward
    Dropout(0.2),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')               # 1=Good, 0=Bad
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# ── Train ─────────────────────────────────────────────────
history = model.fit(X_train, y_train, epochs=80, batch_size=5,
                    validation_data=(X_test, y_test), verbose=0)

print(f'Train Accuracy: {history.history["accuracy"][-1]*100:.1f}%')
print(f'Test  Accuracy: {history.history["val_accuracy"][-1]*100:.1f}%')

In [ ]:
# ── Accuracy Plot ─────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'],     label='Train')
plt.plot(history.history['val_accuracy'], label='Test', linestyle='--')
plt.title('BiLSTM — Training Accuracy over Epochs')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────
y_pred = (model.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
disp = ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                              display_labels=['Bad','Good'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — BiLSTM Product Classifier')
plt.show()

In [ ]:
# ── Predict New Products ───────────────────────────────────
products = [
    {'name': 'Phone A',   'scores': [9, 8, 9, 9]},
    {'name': 'Headphone', 'scores': [2, 3, 2, 2]},
    {'name': 'Laptop B',  'scores': [8, 7, 8, 8]},
    {'name': 'Charger X', 'scores': [3, 4, 3, 2]},
]

print('─' * 55)
for p in products:
    arr  = np.array(p['scores'], dtype='float32') / 10.0
    prob = model.predict(arr.reshape(1, 4, 1), verbose=0)[0][0]
    verdict = 'GOOD ✅' if prob > 0.5 else 'BAD ❌'
    print(f"{p['name']:12s} {p['scores']} → {verdict} ({prob*100:.0f}% Good)")
print('─' * 55)